# Gate-2 text fp16 residual diagnostic (tiny, GPU)

Characterizes the text pooled-vector residual (my-vs-frozen mean cosine 0.99998 /
worst 0.9991) after storage and library version were ruled out. On a 256-text
sample it measures, with NO full extraction or dataset save:
- **run-to-run** fp16 determinism (batch 256 twice), and
- the **batch effect** (batch 256 vs 64 -- the frozen extractor used 64).

Attach the canonical sharded dataset + the checkpoint; GPU, Internet, GITHUB_TOKEN.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '3a3e75386b2e54e5cce3407a38a0e35c01af067c'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
SAMPLE = 256

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as h:
    h.write("#!/usr/bin/env python3\nimport os, sys\np = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in p else 'x-access-token')\n")
os.chmod(askpass, 0o700)
cenv = os.environ.copy(); cenv.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for p in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(p): shutil.rmtree(p)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=cenv)
finally:
    os.remove(askpass); del github_token, cenv
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.4'], check=True)
sys.path.insert(0, WORKTREE)
print('setup PASS')

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)[0]
state = hashlib.sha256()
with open(checkpoint, 'rb') as h:
    for b in iter(lambda: h.read(8 * 1024 * 1024), b''): state.update(b)
assert state.hexdigest() == CHECKPOINT_SHA256, 'checkpoint SHA mismatch'
print({'dataset_root': dataset_root, 'checkpoint': checkpoint})

In [ ]:
import numpy as np
from evaluation.extract_frozen_glim_text_tokens import GLIMTextTokenEmbedder, load_rows, build_text_records
rows, _ = load_rows(Path(dataset_root), EXPECTED_INDEX_SHA256)
records, _ = build_text_records(rows)
texts = [r['representative_text'] for r in records[:SAMPLE]]
print('sample texts:', len(texts))

def cos(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True).clip(1e-12)
    b = b / np.linalg.norm(b, axis=1, keepdims=True).clip(1e-12)
    c = (a * b).sum(1)
    return {'min': round(float(c.min()), 8), 'mean': round(float(c.mean()), 8),
            'max_abs_diff': round(float(np.abs((a - b)).max()), 6)}

emb = GLIMTextTokenEmbedder(Path(GLIM_WORKTREE), Path(checkpoint), 'cuda', text_batch_size=SAMPLE)
_, _, A = emb(texts)          # batch 256
_, _, B = emb(texts)          # batch 256 again -> run-to-run determinism
emb.text_batch_size = 64
_, _, C = emb(texts)          # batch 64 -> the frozen extractor's batch
run_to_run = cos(A, B)
batch_effect = cos(A, C)
print('run-to-run  (256 vs 256):', run_to_run)
print('batch effect (256 vs 64):', batch_effect)
print('reference: my-vs-frozen residual was mean 0.99998 / worst 0.9991')

In [ ]:
rr, be = run_to_run['min'], batch_effect['min']
if be < 0.9995 and rr > 0.99999:
    verdict = ('BATCH is the cause: batch 256-vs-64 reproduces the residual and fp16 is '
               'deterministic run-to-run. Re-extract at batch 64 (the default now) -> expect '
               'the identity check to pass the tight 0.9995 bar.')
elif rr < 0.99999:
    verdict = ('IRREDUCIBLE fp16: even same-config run-to-run differs. The 0.99998 residual is '
               'the fp16 T5 compute floor. Pre-declare the text threshold at this measured floor.')
else:
    verdict = ('NEITHER batch nor run-to-run explains it (both ~1.0). A config difference vs the '
               'frozen extractor remains -- do NOT relax; investigate tokenizer/model-revision/pooling.')
for p in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(p): shutil.rmtree(p)
print('VERDICT:', verdict)